In [14]:
import os
import pandas as pd
import numpy as np

def normalize_df(df, text_col, label_col):
    df = df[[text_col, label_col]].copy()
    df.columns = ["text", "label"]
    df["text"] = df["text"].astype(str)
    # Ép kiểu nhãn về số, các giá trị lỗi sẽ thành NaN
    df["label"] = pd.to_numeric(df["label"], errors="coerce")
    df.dropna(subset=["text", "label"], inplace=True)
    df["label"] = df["label"].astype(int)
    return df

df_list = []
print("=== QUY ƯỚC ĐỒNG NHẤT: 0 = REAL, 1 = FAKE ===")

=== QUY ƯỚC ĐỒNG NHẤT: 0 = REAL, 1 = FAKE ===


In [15]:
print("\n1. Đang xử lý Dataset 1 (Vn-FakeNews local)...")
try:
    # 1. Load file CSV
    df1 = pd.read_csv("datasetvn/Fake News Vietnamese Dataset/public_train.csv")
    df1.rename(columns={'post_message': 'text'}, inplace=True)
    
    # VÌ BẠN KIỂM CHỨNG 0=REAL, 1=FAKE NÊN KHÔNG DÙNG .map ĐẢO NGƯỢC NỮA
    # Chỉ cần đảm bảo cột label là kiểu số nguyên
    df1['label'] = pd.to_numeric(df1['label'], errors='coerce')
    
    df_list.append(normalize_df(df1, "text", "label"))
    
    # 2. Load file Excel
    df1_ex = pd.read_excel("datasetvn/Fake News Vietnamese Dataset/warmup_training_dataset.xlsx")
    df1_ex.rename(columns={'post_message': 'text'}, inplace=True)
    
    # Giữ nguyên nhãn gốc
    df1_ex['label'] = pd.to_numeric(df1_ex['label'], errors='coerce')
    
    df_list.append(normalize_df(df1_ex, "text", "label"))
    print("   ✅ Xong Dataset 1 (Giữ nguyên nhãn 0=Real, 1=Fake)")
except Exception as e: 
    print(f"❌ Lỗi Dataset 1: {e}")


1. Đang xử lý Dataset 1 (Vn-FakeNews local)...
   ✅ Xong Dataset 1 (Giữ nguyên nhãn 0=Real, 1=Fake)


In [16]:
print("\n2 & 3. Đang xử lý nhóm Update Data...")
try:
    path_up = "datasetvn/last_data/update_data/"
    for f in os.listdir(path_up):
        if f.endswith(".csv"):
            temp = pd.read_csv(os.path.join(path_up, f))
            # Đổi tên cột linh hoạt
            temp.rename(columns={'post_message': 'text', 'Maintext': 'text', 'maintext': 'text', 'Label': 'label'}, inplace=True)
            
            # Đồng nhất nhãn nếu nhãn là chữ
            if temp['label'].dtype == 'object':
                mapping = {'Real': 0, 'TIN THẬT': 0, 'Fake': 1, 'TIN GIẢ': 1}
                temp['label'] = temp['label'].str.upper().map(mapping)
            
            df_list.append(normalize_df(temp, "text", "label"))
            print(f"   - Đã nạp: {f}")
except Exception as e: print(f"❌ Lỗi Dataset 2&3: {e}")


2 & 3. Đang xử lý nhóm Update Data...
   - Đã nạp: fix_test_data.csv
   - Đã nạp: update_train_data.csv
   - Đã nạp: update_val_data.csv


In [17]:
print("\n4. Đang xử lý Dataset 4 (Medical)...")
try:
    df_med = pd.read_csv("datasetvn/Vietnamese medical fake news dataset/full_dataset.csv")
    # Quy ước gốc: is_fake (True=Fake, False=Real) -> Map: True:1, False:0 (Khớp chuẩn)
    df_med.rename(columns={'normalized_content': 'text', 'is_fake': 'label'}, inplace=True)
    df_med['label'] = df_med['label'].map({True: 1, False: 0, 'True': 1, 'False': 0})
    df_list.append(normalize_df(df_med, "text", "label"))
    print("   ✅ Xong Dataset 4")
except Exception as e: print(f"❌ Lỗi Dataset 4: {e}")


4. Đang xử lý Dataset 4 (Medical)...
   ✅ Xong Dataset 4


In [18]:
print("\n5. Đang xử lý Dataset 5 (Real News Only)...")
try:
    path5 = "datasetvn/Vietnamese_fake_news_dataset2/"
    for f in ["real_news_5500_balanced.csv", "real_news.csv"]:
        file_p = os.path.join(path5, f)
        if os.path.exists(file_p):
            temp = pd.read_csv(file_p)
            temp.rename(columns={'text': 'text', 'content': 'text'}, inplace=True)
            temp['label'] = 0 # Luôn là tin thật
            df_list.append(normalize_df(temp, "text", "label"))
            print(f"   - Đã nạp: {f}")
except Exception as e: print(f"❌ Lỗi Dataset 5: {e}")


5. Đang xử lý Dataset 5 (Real News Only)...
   - Đã nạp: real_news_5500_balanced.csv
   - Đã nạp: real_news.csv


In [19]:
print("\n2. Đang xử lý Dataset 2 (VFND - JSON structure)...")
import os
import json
from pathlib import Path

try:
    vfnd_data = []
    # Đường dẫn tới thư mục gốc của dataset VFND
    root_path = Path("datasetvn/VFND-vietnamese-fake-news-datasets-master")
    
    # Định nghĩa ánh xạ nhãn chỉ còn Fake và Real
    # 0 = Real, 1 = Fake
    label_map = {
        "Fake": 1,
        "Real": 0
    }

    # Duyệt qua các thư mục nhãn
    for folder_name, label_value in label_map.items():
        folder_path = root_path / folder_name
        
        # Kiểm tra thư mục có tồn tại không trước khi quét
        if not folder_path.exists():
            print(f"   ⚠️ Thư mục {folder_name} không tìm thấy, bỏ qua.")
            continue
            
        # Tìm tất cả file .json trong các thư mục con (Article_Contents, Social_Contents)
        # Quét đệ quy bằng glob
        json_files = list(folder_path.glob("**/*.json"))
        
        for file_path in json_files:
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = json.load(f)
                    
                # Trích xuất nội dung từ trường 'text'
                text_content = content.get("text", "")
                
                # Nếu không có 'text' thì lấy 'title' làm nội dung thay thế
                if not text_content and "title" in content:
                    text_content = content.get("title", "")
                
                # Chỉ thêm vào danh sách nếu có nội dung văn bản
                if text_content and len(str(text_content).strip()) > 0:
                    vfnd_data.append({
                        "text": text_content,
                        "label": label_value
                    })
            except Exception:
                # Bỏ qua nếu file json bị lỗi định dạng hoặc không đọc được
                continue

    # Chuyển đổi list thành DataFrame và đưa vào df_list chung
    if vfnd_data:
        df2 = pd.DataFrame(vfnd_data)
        # Sử dụng hàm normalize_df của bạn để đồng nhất tên cột và kiểu dữ liệu
        df_list.append(normalize_df(df2, "text", "label"))
        print(f"   ✅ Xong Dataset 2: Load thành công {len(df2)} mẫu từ JSON (Fake & Real).")
    else:
        print("   ⚠️ Không tìm thấy dữ liệu hợp lệ trong thư mục Fake và Real.")

except Exception as e:
    print(f"❌ Lỗi nghiêm trọng khi xử lý Dataset 2 (VFND): {e}")


2. Đang xử lý Dataset 2 (VFND - JSON structure)...
   ✅ Xong Dataset 2: Load thành công 276 mẫu từ JSON (Fake & Real).


In [20]:
print("TỔNG HỢP VÀ THỐNG KÊ")

# 1. Gộp toàn bộ list dataframe
df_final = pd.concat(df_list, ignore_index=True)

# 2. Làm sạch cơ bản
df_final.drop_duplicates(subset=['text'], keep='first', inplace=True)
df_final = df_final[df_final['text'].str.len() > 20]

#(B) Chuẩn hóa text nhẹ (Xử lý khoảng trắng thừa)
df_final['text'] = (
    df_final['text']
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

#(A) Shuffle dataset (Xáo trộn để tránh bias thứ tự nguồn tin)
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

# 3. Tính toán thống kê
counts = df_final['label'].value_counts()
percents = df_final['label'].value_counts(normalize=True) * 100

#(C) Log thống kê vào biến (Dùng cho báo cáo hoặc cell sau)
stats = {
    "total": len(df_final),
    "real": counts.get(0, 0),
    "fake": counts.get(1, 0),
    "real_pct": percents.get(0, 0),
    "fake_pct": percents.get(1, 0)
}

print("-" * 30)
print(f"✅ Tổng số dòng: {stats['total']}")
print(f"🔹 Tin Thật (0): {stats['real']} mẫu ({stats['real_pct']:.2f}%)")
print(f"🔸 Tin Giả (1): {stats['fake']} mẫu ({stats['fake_pct']:.2f}%)")
print("-" * 30)

# 4. Lưu file
os.makedirs("ML/data", exist_ok=True)
df_final.to_csv("ML/data/vn_fake_news_combined.csv", index=False)
print("Đã lưu dataset sạch tại: ML/data/vn_fake_news_combined.csv")

# 5. Gán sang biến df
df = df_final.copy() 
print("Biến 'df' đã sẵn sàng cho Training.")
# Hiển thị biến stats để xem định dạng dictionary
stats

TỔNG HỢP VÀ THỐNG KÊ
------------------------------
✅ Tổng số dòng: 22929
🔹 Tin Thật (0): 16517 mẫu (72.04%)
🔸 Tin Giả (1): 6412 mẫu (27.96%)
------------------------------
Đã lưu dataset sạch tại: ML/data/vn_fake_news_combined.csv
Biến 'df' đã sẵn sàng cho Training.


{'total': 22929,
 'real': np.int64(16517),
 'fake': np.int64(6412),
 'real_pct': np.float64(72.03541366828034),
 'fake_pct': np.float64(27.964586331719655)}

In [21]:
# Kiểm tra xem biến nào đang giữ dữ liệu (thường là 'df' hoặc 'df_final')
# Ở đây tôi dùng 'df' theo log quy trình gộp của bạn

try:
    print("📊 Thống kê nhanh nhãn:")
    print(df['label'].value_counts())
    
    print("\n✅ Mẫu tin được gán nhãn REAL (0):")
    display(df[df['label'] == 0]['text'].sample(5))

    print("\n❌ Mẫu tin được gán nhãn FAKE (1):")
    display(df[df['label'] == 1]['text'].sample(5))
except NameError:
    print("❌ Lỗi: Biến 'df' chưa được định nghĩa. Hãy chạy lại Cell tổng hợp dữ liệu trước!")
except Exception as e:
    print(f"❌ Có lỗi xảy ra: {e}")

📊 Thống kê nhanh nhãn:
label
0    16517
1     6412
Name: count, dtype: int64

✅ Mẫu tin được gán nhãn REAL (0):


15736    Huấn Hoa Hồng ra MV ca nhạc phản đối việc cờ b...
14971    Hà_NộiAnh Tuân 38 sốt_xuất_huyết 4 ho máu bác_...
11038    Tính đến 6 giờ sáng 22/4/2020, Bộ Y tế không g...
22461    Trời se lạnh thói_quen uống ấm pha chanh mật_o...
20697    Não mô cầu_là bệnh nhiễm_trùng cấp_tính nghiêm...
Name: text, dtype: str


❌ Mẫu tin được gán nhãn FAKE (1):


15796    Đúng chín uống sôi nấu nhừ vị_giác chất dinh_d...
20875    Bệnh_viện Bệnh Nhiệt_đới Trung_ương Hà_Nội điề...
21557    Trong bếp thời xưa treo sẵn chùm bồ_kết Ngoài ...
7816     Bệnh_nhân BTTTh 37 Tp Cần_Thơ gia_đình bệnh_vi...
21436    Sau tiệc cưới huyện Phong_Điền Thừa_Thiên_Huế ...
Name: text, dtype: str